# Hilvan M5.3 — focal mechanism and Coulomb stress from the **actual downloaded dataset**

This notebook is tailored to the archive you supplied:

`us6000tx9u_20260924_074058_20260924101757454`

The dataset itself reports:

- Event ID: **us6000tx9u**
- Origin: **2026-09-24T07:40:58.138Z**
- Latitude: **37.4736°N**
- Longitude: **38.8612°E**
- Depth: **10 km**
- Magnitude: **5.3**
- Download radius in `study.json`: **200 km**
- Download window: **07:35:58.138–08:40:58.138 UTC**
- Original waveform archive includes MiniSEED, converted SAC, StationXML, station geometry and SeismicDesk analysis files.
- The supplied `analysis/inputs/arrivals.csv` and `analysis/inputs/polarities.csv` are empty, so the focal mechanism has not yet been solved.

## What this notebook does

1. Opens either the ZIP archive **or** an already-extracted study folder.
2. Reads `study.json`, `channels.csv`, station geometry and StationXML.
3. Uses **original MiniSEED preferentially**, avoiding duplicate SAC conversions.
4. Keeps only traces that actually contain the event/P arrival.
5. Selects the best vertical channel for each physical station site.
6. Deduplicates co-located duplicate providers/networks.
7. Predicts P arrivals with the same default velocity model used by the supplied analysis (`iasp91`).
8. Refines P picks automatically.
9. Deconvolves instrument response to velocity when StationXML permits it.
10. Generates automatic P first-motion polarity candidates and QC plots.
11. Supports manual review before a final solution.
12. Inverts P polarities for **strike, dip, rake** using a double-couple grid search consistent with the supplied SeismicDesk focal-mechanism equations.
13. Computes NP1 and NP2 and a beachball.
14. Estimates a finite rectangular source from magnitude and an assumed stress drop.
15. Computes Okada elastic-half-space Coulomb failure stress for **both nodal planes**.
16. Exports all mechanism, QC and Coulomb results.

> **Important:** automatic first-motion polarity is provisional until visually reviewed. Do not publish or interpret the Coulomb map as a final geological result before checking the P polarities and independently deciding which nodal plane is the physical rupture plane.

### Repository use
This notebook reproduces the focal-mechanism and Coulomb workflow from the original downloaded
study archive. The raw archive is not redistributed here. Put the original/re-downloaded study ZIP
inside `data/raw/` or set `DATA_SOURCE` manually. The exact derived tables and grids used in the
paper are already archived under `data/derived/study_snapshot/focal_coulomb_results/`.

## 0. Package setup

For the waveform/focal section the main requirements are ObsPy, NumPy, SciPy, pandas and matplotlib.

The Coulomb section uses **Pyrocko's Okada implementation**. If Pyrocko cannot be installed by pip on your Windows environment, install it in a conda environment using:

```bash
conda install -c pyrocko pyrocko
```

In [ ]:
import sys, subprocess, importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "obspy": "obspy>=1.4.2",
    "pyproj": "pyproj",
    "tqdm": "tqdm",
}

missing = [pkg for mod, pkg in required.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("Core packages are available.")

if importlib.util.find_spec("pyrocko") is None:
    print("\nPyrocko is not installed.")
    print("Trying pip install pyrocko ...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pyrocko"])
    except Exception as exc:
        print("Pyrocko pip installation did not succeed:", exc)
        print("For Windows/Conda use: conda install -c pyrocko pyrocko")

# Part I — load the actual downloaded study

## 1. Point the notebook to your ZIP or extracted folder

You normally only need to edit `DATA_SOURCE`.

Examples:

```python
DATA_SOURCE = r"/path/to/my_hilvan_data.zip"
```

or

```python
DATA_SOURCE = r"/path/to/us6000tx9u_20260924_074058_20260924101757454"
```

If it is `None`, the notebook searches the current directory for a ZIP/folder containing `study.json`.

In [ ]:
from pathlib import Path
import zipfile, shutil, json, math, warnings, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from obspy import UTCDateTime, Stream, read, read_inventory
from obspy.taup import TauPyModel
from obspy.signal.trigger import classic_sta_lta, trigger_onset
from obspy.geodetics import gps2dist_azimuth
from obspy.imaging.beachball import beach, aux_plane

# ---------------- REPOSITORY-AWARE INPUT ----------------
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

raw_dir = REPO_ROOT / "data" / "raw"
candidate_zips = sorted(raw_dir.glob("*.zip")) if raw_dir.exists() else []
DATA_SOURCE = candidate_zips[0] if candidate_zips else None
# Or set explicitly:
# DATA_SOURCE = Path(r"/path/to/your/downloaded_hilvan_data.zip")
# ----------------------------------------------------------

WORK_BASE = REPO_ROOT / "work" / "Hilvan_M53_Focal_Coulomb_Work"
WORK_BASE.mkdir(parents=True, exist_ok=True)

def is_study_folder(p):
    p = Path(p)
    return p.is_dir() and (p / "study.json").exists()

def zip_contains_study(p):
    try:
        with zipfile.ZipFile(p) as z:
            return any(n.endswith("/study.json") or n == "study.json" for n in z.namelist())
    except Exception:
        return False

def auto_find_source():
    cwd = Path.cwd()

    folders = [p for p in cwd.iterdir() if is_study_folder(p)]
    preferred = [p for p in folders if p.name.startswith("us6000tx9u_")]
    if preferred:
        return preferred[0]
    if folders:
        return folders[0]

    zips = [p for p in cwd.glob("*.zip") if zip_contains_study(p)]
    preferred = [p for p in zips if "us6000tx9u" in p.name.lower()]
    if preferred:
        return preferred[0]
    if len(zips) == 1:
        return zips[0]
    if zips:
        return zips[0]
    return None

source = Path(DATA_SOURCE) if DATA_SOURCE else auto_find_source()

if source is None or not source.exists():
    raise FileNotFoundError(
        "Study ZIP/folder was not found. Set DATA_SOURCE in the cell above."
    )

if source.is_file() and source.suffix.lower() == ".zip":
    extract_dir = WORK_BASE / "extracted_study"
    marker = extract_dir / ".extracted_ok"

    if not marker.exists():
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True)
        print("Extracting:", source)
        with zipfile.ZipFile(source) as z:
            z.extractall(extract_dir)
        marker.write_text("ok", encoding="utf-8")

    study_jsons = list(extract_dir.rglob("study.json"))
    if not study_jsons:
        raise FileNotFoundError("No study.json found after ZIP extraction.")
    STUDY_ROOT = study_jsons[0].parent
else:
    STUDY_ROOT = source

print("Study root:", STUDY_ROOT)

## 2. Read the event and existing analysis metadata

The notebook takes event parameters from the supplied `study.json` instead of hard-coding them.

In [ ]:
study = json.loads((STUDY_ROOT / "study.json").read_text(encoding="utf-8-sig"))
event = study["Event"]

EVENT_ID = str(event["Id"])
EVENT_TIME = UTCDateTime(event["Time"])
EVENT_LAT = float(event["Latitude"])
EVENT_LON = float(event["Longitude"])
EVENT_DEPTH_KM = float(event["Depth"])
EVENT_MAG = float(event["Magnitude"])

DOWNLOAD_RADIUS_KM = float(study.get("RadiusKm", np.nan))
DOWNLOAD_START = UTCDateTime(study["Start"])
DOWNLOAD_END = UTCDateTime(study["End"])

ANALYSIS_ROOT = STUDY_ROOT / "analysis"
OUT = STUDY_ROOT / "focal_coulomb_results"
TABLES = OUT / "tables"
FIGS = OUT / "figures"
CFS_DIR = OUT / "coulomb"

for p in [OUT, TABLES, FIGS, CFS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

config_path = ANALYSIS_ROOT / "analysis-config.json"
analysis_config = json.loads(config_path.read_text(encoding="utf-8-sig")) if config_path.exists() else {}

VELOCITY_MODEL = analysis_config.get("VelocityModel", "iasp91")

print("Event:", EVENT_ID)
print("Origin:", EVENT_TIME)
print("Location:", EVENT_LAT, EVENT_LON)
print("Depth km:", EVENT_DEPTH_KM)
print("Magnitude:", EVENT_MAG)
print("Downloaded radius km:", DOWNLOAD_RADIUS_KM)
print("Velocity model:", VELOCITY_MODEL)

## 3. Audit the archive

The supplied dataset contains both original MiniSEED and SAC conversions. The SAC conversion log says raw samples were converted **without response removal**, and the supplied README states paired SAC conversions should be ignored when their original MiniSEED exists.

Therefore this notebook uses MiniSEED as the primary waveform source.

In [ ]:
mseed_files = sorted(STUDY_ROOT.rglob("*.mseed"))
sac_files = sorted(STUDY_ROOT.rglob("*.sac"))
stationxml_files = sorted(STUDY_ROOT.rglob("response.stationxml"))

geo_path = ANALYSIS_ROOT / "02_station_metadata" / "station_geometry.csv"
channels_path = STUDY_ROOT / "channels.csv"

geo = pd.read_csv(geo_path) if geo_path.exists() else pd.DataFrame()
channels = pd.read_csv(channels_path) if channels_path.exists() else pd.DataFrame()

print("MiniSEED files:", len(mseed_files))
print("SAC files:", len(sac_files))
print("StationXML files:", len(stationxml_files))
print("Station geometry rows:", len(geo))
print("Channel rows:", len(channels))

if not geo.empty:
    display(
        geo[["Provider","Network","Station","EpicentralKm","AzimuthDeg",
             "Channels","SampleRatesHz"]]
        .sort_values("EpicentralKm")
    )

# Part II — select scientifically useful event waveforms

## 4. Read only MiniSEED traces that can contain this earthquake

The archive has an approximately one-hour event block and a later short block.  
For focal analysis we only keep traces that overlap the origin-to-P/S event interval.

Vertical-component preference:

1. `HHZ`
2. `BHZ`
3. `EHZ`
4. `SHZ`
5. `HNZ` only as fallback, because it is normally a strong-motion accelerometer.

Co-located duplicate providers are later reduced to one physical site.

In [ ]:
# Analysis controls
MAX_FOCAL_DISTANCE_KM = 200.0
ALLOW_HNZ_FALLBACK = True
SITE_DEDUP_KM = 0.75

P_FILTER_LOW = 0.5
P_FILTER_HIGH = 10.0

P_SEARCH_BEFORE = 5.0
P_SEARCH_AFTER = 8.0

MIN_AUTO_SNR = 5.0
MIN_AUTO_CONFIDENCE = 3.5

taup = TauPyModel(model=VELOCITY_MODEL)

# Cache inventory next to every waveform file.
inventory_cache = {}

def load_local_inventory(mseed_path):
    parent = Path(mseed_path).parent
    if parent in inventory_cache:
        return inventory_cache[parent]
    xp = parent / "response.stationxml"
    if not xp.exists():
        inventory_cache[parent] = None
        return None
    try:
        inventory_cache[parent] = read_inventory(str(xp))
    except Exception:
        inventory_cache[parent] = None
    return inventory_cache[parent]

def geo_row(net, sta):
    if geo.empty:
        return None
    q = geo[
        (geo["Network"].astype(str) == str(net))
        & (geo["Station"].astype(str) == str(sta))
    ]
    return None if q.empty else q.iloc[0]

def earliest_p(distance_degrees):
    arrivals = taup.get_travel_times(
        source_depth_in_km=max(0.0, EVENT_DEPTH_KM),
        distance_in_degree=float(distance_degrees),
        phase_list=["P","p","Pn","Pg"]
    )
    if not arrivals:
        return None
    return min(arrivals, key=lambda a: a.time)

def channel_rank(code):
    code = str(code).upper()
    order = {
        "HHZ": 100,
        "BHZ": 90,
        "EHZ": 80,
        "SHZ": 70,
        "HNZ": 45,
    }
    return order.get(code, 10 if code.endswith("Z") else 0)

records = []

for mf in tqdm(mseed_files, desc="Scanning event MiniSEED"):
    inv = load_local_inventory(mf)

    try:
        st = read(str(mf), details=True)
    except Exception as exc:
        continue

    for tr in st:
        if not tr.stats.channel.upper().endswith("Z"):
            continue

        gr = geo_row(tr.stats.network, tr.stats.station)
        if gr is None:
            continue

        dist_km = float(gr["EpicentralKm"])
        if dist_km > MAX_FOCAL_DISTANCE_KM:
            continue

        arrival = earliest_p(float(gr["DistanceDegrees"]))
        if arrival is None:
            continue

        p_pred = EVENT_TIME + float(arrival.time)

        # Only traces/segments that include the P region.
        if tr.stats.endtime < p_pred + 2.0 or tr.stats.starttime > p_pred - 3.0:
            continue

        orientation_ok = None
        dip = np.nan
        sensitivity = np.nan

        if inv is not None:
            try:
                ori = inv.get_orientation(tr.id, tr.stats.starttime)
                dip = float(ori["dip"])
                orientation_ok = abs(dip + 90.0) <= 5.0
            except Exception:
                pass

            try:
                resp = inv.get_response(tr.id, tr.stats.starttime)
                sensitivity = float(resp.instrument_sensitivity.value)
            except Exception:
                pass

        records.append({
            "file": str(mf),
            "provider_folder": mf.parents[1].name if len(mf.parents) > 1 else "",
            "id": tr.id,
            "network": tr.stats.network,
            "station": tr.stats.station,
            "location": tr.stats.location,
            "channel": tr.stats.channel,
            "sample_rate_hz": float(tr.stats.sampling_rate),
            "start": str(tr.stats.starttime),
            "end": str(tr.stats.endtime),
            "latitude": float(gr["Latitude"]),
            "longitude": float(gr["Longitude"]),
            "epicentral_km": dist_km,
            "distance_deg": float(gr["DistanceDegrees"]),
            "azimuth_deg": float(gr["AzimuthDeg"]),
            "backazimuth_deg": float(gr["BackAzimuthDeg"]),
            "predicted_phase": arrival.name,
            "predicted_p_time": str(p_pred),
            "takeoff_angle_deg": float(arrival.takeoff_angle),
            "incident_angle_deg": float(arrival.incident_angle),
            "channel_dip_deg": dip,
            "vertical_orientation_ok": orientation_ok,
            "instrument_sensitivity": sensitivity,
            "_trace": tr,
            "_inventory": inv,
        })

candidate_df = pd.DataFrame([
    {k:v for k,v in r.items() if not k.startswith("_")}
    for r in records
])

print("Vertical trace segments containing predicted P:", len(candidate_df))
display(candidate_df.sort_values(["epicentral_km","network","station","channel"]).head(80))

## 5. Response-correct each candidate and score waveform quality

For consistent polarity across seismometers and accelerometers, the notebook attempts to remove the instrument response to **ground velocity (m/s)**.

The automatic polarity is accepted only if:
- StationXML response correction succeeds;
- the Z orientation is standard (approximately dip = −90°);
- signal-to-noise and first-motion confidence pass thresholds;
- the polarity is stable across two reasonable P-wave passbands.

Uncorrected data remain available for inspection but are not automatically trusted for final polarity.

In [ ]:
from scipy.signal import detrend as scipy_detrend

def safe_remove_response(tr, inv):
    x = tr.copy()
    if inv is None:
        return None, "no_stationxml"

    try:
        x.detrend("demean")
        x.detrend("linear")
        x.taper(max_percentage=0.03)

        nyq = 0.5 * x.stats.sampling_rate
        hi2 = min(20.0, 0.80*nyq)
        hi1 = min(15.0, 0.65*nyq)
        if hi1 <= 0.2 or hi2 <= hi1:
            return None, "sampling_too_low"

        pre_filt = (0.03, 0.08, hi1, hi2)

        x.remove_response(
            inventory=inv,
            output="VEL",
            pre_filt=pre_filt,
            water_level=60,
            zero_mean=False,
            taper=False,
        )
        return x, "ok"

    except Exception as exc:
        return None, str(exc)

def preprocess_pick(tr, fmin=0.5, fmax=10.0):
    x = tr.copy()
    x.detrend("demean")
    x.detrend("linear")
    x.taper(max_percentage=0.02)

    nyq = 0.5*x.stats.sampling_rate
    high = min(float(fmax), 0.80*nyq)

    if high > fmin:
        x.filter(
            "bandpass",
            freqmin=float(fmin),
            freqmax=float(high),
            corners=3,
            zerophase=True
        )
    return x

def rms(a):
    a = np.asarray(a, float)
    return float(np.sqrt(np.mean(a*a))) if len(a) else np.nan

def refine_pick(tr_vel, predicted):
    x = preprocess_pick(tr_vel, P_FILTER_LOW, P_FILTER_HIGH)
    w = x.copy().trim(
        predicted-P_SEARCH_BEFORE,
        predicted+P_SEARCH_AFTER,
        pad=False
    )

    fs = float(w.stats.sampling_rate)
    if len(w.data) < max(50, int(3*fs)):
        return predicted, "taup"

    nsta = max(2, int(0.20*fs))
    nlta = max(nsta+1, int(1.50*fs))

    try:
        cft = classic_sta_lta(np.asarray(w.data, float), nsta, nlta)
        pairs = trigger_onset(cft, 2.8, 1.1)

        choices = []
        for on, off in pairs:
            t = w.stats.starttime + on/fs
            # restrict automatic refinement near the predicted phase
            if predicted-3.0 <= t <= predicted+5.0:
                choices.append((abs(t-predicted), t))

        if choices:
            choices.sort(key=lambda z:z[0])
            return choices[0][1], "sta_lta"
    except Exception:
        pass

    return predicted, "taup"

def first_motion_metrics(tr_vel, pick, fmin, fmax):
    x = preprocess_pick(tr_vel, fmin, fmax)

    noise = x.copy().trim(pick-4.0, pick-0.7, pad=False)
    signal = x.copy().trim(pick-0.08, pick+0.70, pad=False)

    if len(noise.data) < 10 or len(signal.data) < 5:
        return None

    n = np.asarray(noise.data, float)
    s = np.asarray(signal.data, float)

    baseline = np.median(n)
    n = n-baseline
    s = s-baseline

    nrms = rms(n)
    if not np.isfinite(nrms) or nrms <= 0:
        return None

    peak = float(np.max(np.abs(s)))
    snr = peak/nrms

    threshold = max(3.0*nrms, 0.10*peak)
    ids = np.where(np.abs(s) >= threshold)[0]

    if len(ids) == 0:
        return dict(polarity=0, snr=snr, confidence=0.0)

    i = int(ids[0])
    fs = float(signal.stats.sampling_rate)
    h = max(1, int(0.015*fs))
    j1, j2 = max(0, i-h), min(len(s), i+h+1)
    amp = float(np.mean(s[j1:j2]))

    pol = 1 if amp > 0 else -1
    conf = abs(amp)/nrms

    return dict(
        polarity=pol,
        snr=snr,
        confidence=conf,
        first_amp=amp,
        threshold=threshold
    )

In [ ]:
quality_rows = []
processed = {}

for i, r in enumerate(tqdm(records, desc="Response correction + P polarity")):
    tr = r["_trace"]
    inv = r["_inventory"]
    corrected, status = safe_remove_response(tr, inv)

    if corrected is None:
        quality_rows.append({
            "record_index": i,
            "id": r["id"],
            "network": r["network"],
            "station": r["station"],
            "channel": r["channel"],
            "epicentral_km": r["epicentral_km"],
            "azimuth_deg": r["azimuth_deg"],
            "takeoff_angle_deg": r["takeoff_angle_deg"],
            "response_status": status,
            "accepted_auto": False,
        })
        continue

    predicted = UTCDateTime(r["predicted_p_time"])
    pick, pick_method = refine_pick(corrected, predicted)

    m1 = first_motion_metrics(corrected, pick, 0.5, 5.0)
    m2 = first_motion_metrics(corrected, pick, 1.0, 8.0)

    if m1 is None or m2 is None:
        accepted = False
        consensus = 0
        snr = np.nan
        confidence = np.nan
    else:
        same = (m1["polarity"] != 0 and m1["polarity"] == m2["polarity"])
        consensus = int(m1["polarity"]) if same else 0
        snr = min(float(m1["snr"]), float(m2["snr"]))
        confidence = min(float(m1["confidence"]), float(m2["confidence"]))

        standard_vertical = (
            r["vertical_orientation_ok"] is True
            or (np.isfinite(r["channel_dip_deg"]) and abs(r["channel_dip_deg"]+90) <= 5)
        )

        acceptable_family = (
            not r["channel"].upper().startswith("HN")
            or ALLOW_HNZ_FALLBACK
        )

        accepted = (
            same
            and snr >= MIN_AUTO_SNR
            and confidence >= MIN_AUTO_CONFIDENCE
            and standard_vertical
            and acceptable_family
        )

    key = f"{r['provider_folder']}|{r['id']}|{r['start']}"
    processed[key] = corrected

    quality_rows.append({
        "record_index": i,
        "record_key": key,
        "provider_folder": r["provider_folder"],
        "id": r["id"],
        "network": r["network"],
        "station": r["station"],
        "location": r["location"],
        "channel": r["channel"],
        "sample_rate_hz": r["sample_rate_hz"],
        "latitude": r["latitude"],
        "longitude": r["longitude"],
        "epicentral_km": r["epicentral_km"],
        "distance_deg": r["distance_deg"],
        "azimuth_deg": r["azimuth_deg"],
        "takeoff_angle_deg": r["takeoff_angle_deg"],
        "channel_dip_deg": r["channel_dip_deg"],
        "predicted_p_time": r["predicted_p_time"],
        "refined_p_time": str(pick),
        "pick_method": pick_method,
        "auto_polarity": consensus,
        "snr": snr,
        "confidence": confidence,
        "response_status": status,
        "accepted_auto": bool(accepted),
        "source_file": r["file"],
    })

quality = pd.DataFrame(quality_rows)
quality.to_csv(TABLES / "all_vertical_P_polarity_candidates.csv", index=False)

print("Candidate records:", len(quality))
print("Automatic accepted records:", int(quality["accepted_auto"].fillna(False).sum()))
display(
    quality.sort_values(
        ["accepted_auto","confidence","sample_rate_hz"],
        ascending=[False,False,False]
    ).head(80)
)

## 6. Deduplicate co-located physical sites

Your actual archive contains duplicate/co-located data sources, including:
- duplicate GAZ retrievals;
- ARPR/ARPRA instruments at essentially the same physical site.

For a focal mechanism these should not count as independent azimuth observations.

The code clusters sites geographically and retains the best accepted waveform at each site.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dp = np.deg2rad(lat2-lat1)
    dl = np.deg2rad(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return float(2*R*np.arcsin(np.sqrt(a)))

def assign_site_clusters(df, radius_km=SITE_DEDUP_KM):
    centers = []
    labels = []

    for _, row in df.iterrows():
        assigned = None
        for j, (la, lo) in enumerate(centers):
            if haversine_km(row.latitude, row.longitude, la, lo) <= radius_km:
                assigned = j
                break

        if assigned is None:
            centers.append((float(row.latitude), float(row.longitude)))
            assigned = len(centers)-1

        labels.append(assigned)

    out = df.copy()
    out["site_cluster"] = labels
    return out

q = quality.copy()
q = q[np.isfinite(q["latitude"]) & np.isfinite(q["longitude"])].copy()
q = assign_site_clusters(q)

def row_score(r):
    ch = str(r.channel).upper()
    family = (
        100 if ch.startswith("HH") else
        90 if ch.startswith("BH") else
        80 if ch.startswith("EH") else
        70 if ch.startswith("SH") else
        45 if ch.startswith("HN") else 10
    )
    return (
        1000 if bool(r.accepted_auto) else 0,
        family,
        0 if not np.isfinite(r.confidence) else float(r.confidence),
        float(r.sample_rate_hz),
    )

selected_rows = []

for site, g in q.groupby("site_cluster"):
    records_site = [r for _, r in g.iterrows()]
    best_row = max(records_site, key=row_score)
    selected_rows.append(best_row)

site_best = pd.DataFrame(selected_rows).reset_index(drop=True)
site_best.to_csv(TABLES / "best_vertical_record_per_physical_site.csv", index=False)

print("Physical sites represented by event waveforms:", len(site_best))
print("Automatically accepted physical sites:", int(site_best["accepted_auto"].sum()))

display(
    site_best[
        ["site_cluster","provider_folder","network","station","channel",
         "epicentral_km","azimuth_deg","takeoff_angle_deg",
         "auto_polarity","snr","confidence","accepted_auto"]
    ].sort_values("azimuth_deg")
)

## 7. Azimuth-coverage check

The supplied study is unusually useful for a first-motion solution because the event-window stations surround a large part of the focal sphere.

The formal threshold used by the supplied SeismicDesk analysis is a maximum azimuth gap below 180°. The notebook recomputes that value **after geographic deduplication**.

In [ ]:
def azimuth_gap_deg(values):
    a = np.sort(np.mod(np.asarray(values, float), 360.0))
    if len(a) < 2:
        return 360.0
    gaps = np.diff(np.r_[a, a[0]+360.0])
    return float(np.max(gaps))

accepted_sites = site_best[site_best["accepted_auto"]].copy()
gap = azimuth_gap_deg(accepted_sites["azimuth_deg"]) if len(accepted_sites) else 360.0

print("Accepted physical sites:", len(accepted_sites))
print(f"Maximum accepted-station azimuth gap: {gap:.1f}°")

if gap >= 180:
    warnings.warn(
        "Azimuth gap is >=180°. Automatic focal inversion is poorly constrained; "
        "manually recover additional polarities before interpreting it."
    )
elif gap >= 160:
    warnings.warn(
        "Azimuth gap is below 180° but still large. Treat the mechanism uncertainty seriously."
    )

# Part III — waveform QC and manual polarity review

## 8. Plot every selected P first motion

Each panel shows:
- the refined P pick at 0 s;
- normalized response-corrected vertical velocity;
- automatic compression (`+1`) or dilatation (`−1`);
- SNR and confidence.

**Inspect these panels carefully.**

In [ ]:
plot_rows = site_best.sort_values("azimuth_deg").copy()

if len(plot_rows):
    ncols = 3
    nrows = int(np.ceil(len(plot_rows)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, max(3.4*nrows, 5)))
    axes = np.atleast_1d(axes).ravel()

    for ax, (_, r) in zip(axes, plot_rows.iterrows()):
        key = r["record_key"]
        tr = processed.get(key)

        if tr is None:
            ax.axis("off")
            continue

        pt = UTCDateTime(r["refined_p_time"])
        x = preprocess_pick(tr, 0.5, 8.0).copy().trim(pt-2.0, pt+4.0)
        tt = x.times(reftime=pt)
        yy = np.asarray(x.data, float)
        sc = np.max(np.abs(yy))
        if sc > 0:
            yy = yy/sc

        ax.plot(tt, yy, linewidth=0.9)
        ax.axvline(0, linewidth=1)
        ax.axhline(0, linewidth=0.5)

        pol = int(r["auto_polarity"]) if np.isfinite(r["auto_polarity"]) else 0
        label = "+1 compression" if pol == 1 else "-1 dilatation" if pol == -1 else "uncertain"

        ax.set_title(
            f'{r["network"]}.{r["station"]} {r["channel"]} | '
            f'{r["epicentral_km"]:.0f} km | az {r["azimuth_deg"]:.0f}°\n'
            f'{label}; SNR={r["snr"]:.1f}; conf={r["confidence"]:.1f}',
            fontsize=9
        )
        ax.set_xlim(-2, 4)
        ax.set_xlabel("s from refined P")

    for ax in axes[len(plot_rows):]:
        ax.axis("off")

    fig.suptitle("Hilvan M5.3 — P first-motion quality control", fontsize=14)
    fig.tight_layout()
    fig.savefig(FIGS / "P_first_motion_QC.png", dpi=220, bbox_inches="tight")
    plt.show()

## 9. Create/reload the editable review table

On first run the notebook creates:

`focal_coulomb_results/tables/POLARITY_REVIEW.csv`

Important columns:
- `use`: 1 include, 0 exclude;
- `reviewed_polarity`: +1 compression, −1 dilatation;
- `reviewed`: TRUE only after you visually confirm it;
- `comment`: optional.

### Exploratory versus final mode

`ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES=True` allows the notebook to continue automatically for an **exploratory** Coulomb map.

Set it to `False` for your final scientific analysis; then only manually reviewed rows are accepted.

If strict automatic QC leaves fewer than 8 stations, the updated code can add carefully filtered lower-confidence stations **only for an exploratory solution**. Those fallback observations are selected to improve azimuth coverage and are strongly down-weighted; the notebook labels the result `EXPLORATORY_UNDERCONSTRAINED_AUTO` when appropriate.

In [ ]:
# -----------------------------------------------------------------------------
# POLARITY REVIEW + CONTROLLED LOW-CONSTRAINT FALLBACK
# -----------------------------------------------------------------------------
# Final/publishable analysis: set ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES = False
# Exploratory analysis: True allows automatic polarities, including a carefully
# down-weighted fallback if the strict QC leaves too few stations.

ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES = True

# Strict focal-mechanism targets. These are quality goals, not absolute laws.
STRICT_TARGET_POLARITIES = 8
STRICT_MAX_AZIMUTH_GAP_DEG = 180.0

# When strict automatic QC leaves too few observations, allow lower-confidence
# candidates only for an EXPLORATORY solution. They must still have:
#   * a valid non-zero automatic polarity
#   * successful StationXML response correction
#   * a near-vertical Z orientation
#   * usable SNR/confidence
AUTO_EXPAND_IF_FEW = True
FALLBACK_MIN_SNR = 2.5
FALLBACK_MIN_CONFIDENCE = 1.25
FALLBACK_MAX_ORIENTATION_ERROR_DEG = 10.0
MAX_EXPLORATORY_POLARITIES = 14
MIN_EXPLORATORY_POLARITIES = 3

# If True, restore the old strict behavior and stop when the quality goals are
# not met. Leave False for exploratory NP1/NP2 + Coulomb scenario generation.
STOP_IF_UNDERCONSTRAINED = False

review_path = TABLES / "POLARITY_REVIEW.csv"

if not review_path.exists():
    review = site_best.copy()
    review["use"] = review["accepted_auto"].astype(int)
    review["reviewed_polarity"] = review["auto_polarity"]
    review["reviewed"] = False
    review["comment"] = ""
    review.to_csv(review_path, index=False)
    print("Created review file:", review_path)
else:
    print("Using existing review file:", review_path)

review = pd.read_csv(review_path)

# Robust type conversion for files edited in Excel.
review["reviewed_bool"] = (
    review.get("reviewed", False).astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes", "y"])
)
review["use"] = pd.to_numeric(review.get("use", 0), errors="coerce").fillna(0).astype(int)
review["reviewed_polarity"] = pd.to_numeric(
    review.get("reviewed_polarity", np.nan), errors="coerce"
)
review["auto_polarity"] = pd.to_numeric(
    review.get("auto_polarity", np.nan), errors="coerce"
)
review["snr"] = pd.to_numeric(review.get("snr", np.nan), errors="coerce")
review["confidence"] = pd.to_numeric(review.get("confidence", np.nan), errors="coerce")
review["azimuth_deg"] = pd.to_numeric(review.get("azimuth_deg", np.nan), errors="coerce")
review["takeoff_angle_deg"] = pd.to_numeric(
    review.get("takeoff_angle_deg", np.nan), errors="coerce"
)
review["channel_dip_deg"] = pd.to_numeric(
    review.get("channel_dip_deg", np.nan), errors="coerce"
)

# -----------------------------------------------------------------------------
# 1) Build the strict starting set.
# -----------------------------------------------------------------------------
if ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES:
    strict_mask = (
        (review["use"] == 1)
        & review["reviewed_polarity"].isin([-1, 1])
        & np.isfinite(review["azimuth_deg"])
        & np.isfinite(review["takeoff_angle_deg"])
    )
    mechanism_input = review[strict_mask].copy()
    mechanism_input["selection_source"] = np.where(
        mechanism_input["reviewed_bool"],
        "manual_review",
        "strict_auto"
    )
    SOLUTION_STATUS = "PROVISIONAL_AUTO_UNLESS_REVIEWED"
else:
    strict_mask = (
        (review["use"] == 1)
        & review["reviewed_bool"]
        & review["reviewed_polarity"].isin([-1, 1])
        & np.isfinite(review["azimuth_deg"])
        & np.isfinite(review["takeoff_angle_deg"])
    )
    mechanism_input = review[strict_mask].copy()
    mechanism_input["selection_source"] = "manual_review"
    SOLUTION_STATUS = "MANUALLY_REVIEWED"

# -----------------------------------------------------------------------------
# 2) Controlled fallback for exploratory analysis only.
#    Candidates are chosen greedily to reduce the maximum azimuth gap first,
#    then by confidence/SNR. This is better than simply taking the next-highest
#    confidence stations clustered in the same direction.
# -----------------------------------------------------------------------------
def _gap_for_df(df):
    if len(df) < 2:
        return 360.0
    vals = pd.to_numeric(df["azimuth_deg"], errors="coerce").dropna().to_numpy(float)
    return azimuth_gap_deg(vals) if len(vals) >= 2 else 360.0

initial_n = len(mechanism_input)
initial_gap = _gap_for_df(mechanism_input)

need_expand = (
    ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES
    and AUTO_EXPAND_IF_FEW
    and (
        len(mechanism_input) < STRICT_TARGET_POLARITIES
        or initial_gap >= STRICT_MAX_AZIMUTH_GAP_DEG
    )
)

fallback_added = []

if need_expand:
    # Keep only physically safer fallback records.
    response_ok = review.get("response_status", "").astype(str).str.lower().eq("ok")
    orientation_ok = (
        np.isfinite(review["channel_dip_deg"])
        & (np.abs(review["channel_dip_deg"] + 90.0) <= FALLBACK_MAX_ORIENTATION_ERROR_DEG)
    )

    used_keys = set(mechanism_input.get("record_key", pd.Series(dtype=str)).astype(str))

    candidates = review[
        review["auto_polarity"].isin([-1, 1])
        & np.isfinite(review["azimuth_deg"])
        & np.isfinite(review["takeoff_angle_deg"])
        & response_ok
        & orientation_ok
        & (review["snr"] >= FALLBACK_MIN_SNR)
        & (review["confidence"] >= FALLBACK_MIN_CONFIDENCE)
    ].copy()

    if "record_key" in candidates.columns and used_keys:
        candidates = candidates[~candidates["record_key"].astype(str).isin(used_keys)]

    # Use the automatic polarity in memory only; do NOT overwrite the review CSV.
    candidates["reviewed_polarity"] = candidates["auto_polarity"].astype(int)
    candidates["selection_source"] = "fallback_auto_for_coverage"

    while len(candidates) and len(mechanism_input) < MAX_EXPLORATORY_POLARITIES:
        current_gap = _gap_for_df(mechanism_input)

        # Stop once both target count and acceptable azimuth gap are reached.
        if (
            len(mechanism_input) >= STRICT_TARGET_POLARITIES
            and current_gap < STRICT_MAX_AZIMUTH_GAP_DEG
        ):
            break

        best_idx = None
        best_rank = None

        for ci, cand in candidates.iterrows():
            trial = pd.concat(
                [mechanism_input, cand.to_frame().T],
                ignore_index=True,
                sort=False
            )
            trial_gap = _gap_for_df(trial)

            # Higher confidence/SNR preferred only after coverage improvement.
            conf = float(cand["confidence"]) if np.isfinite(cand["confidence"]) else 0.0
            snr = float(cand["snr"]) if np.isfinite(cand["snr"]) else 0.0
            quality = conf + 0.04 * min(snr, 50.0)

            # Primary: smallest resulting azimuth gap.
            # Secondary: highest quality.
            rank = (trial_gap, -quality)

            if best_rank is None or rank < best_rank:
                best_rank = rank
                best_idx = ci

        if best_idx is None:
            break

        chosen = candidates.loc[[best_idx]].copy()
        fallback_added.append(chosen.iloc[0])
        mechanism_input = pd.concat(
            [mechanism_input, chosen],
            ignore_index=True,
            sort=False
        )
        candidates = candidates.drop(index=best_idx)

# -----------------------------------------------------------------------------
# 3) Final polarity/weight assignment.
# -----------------------------------------------------------------------------
mechanism_input["polarity"] = pd.to_numeric(
    mechanism_input["reviewed_polarity"], errors="coerce"
).astype(int)

# Base confidence weight.
base_weight = np.clip(
    pd.to_numeric(mechanism_input["confidence"], errors="coerce").fillna(2.0) / 8.0,
    0.15,
    1.0
)

# Manually reviewed rows get full authority; strict auto rows retain normal
# confidence weights; fallback rows are deliberately down-weighted.
mechanism_input["weight"] = base_weight
mechanism_input.loc[
    mechanism_input["selection_source"].eq("manual_review"), "weight"
] = 1.0
mechanism_input.loc[
    mechanism_input["selection_source"].eq("fallback_auto_for_coverage"), "weight"
] = np.clip(
    mechanism_input.loc[
        mechanism_input["selection_source"].eq("fallback_auto_for_coverage"),
        "weight"
    ] * 0.55,
    0.10,
    0.40
)

# Remove any accidental duplicates after concatenation.
dedup_cols = [c for c in ["site_cluster", "network", "station", "channel", "azimuth_deg"]
              if c in mechanism_input.columns]
if dedup_cols:
    mechanism_input = mechanism_input.drop_duplicates(subset=dedup_cols, keep="first")

mechanism_input = mechanism_input.sort_values("azimuth_deg").reset_index(drop=True)

gap_final = _gap_for_df(mechanism_input)

# Refine status label so later figures/CSV files cannot be mistaken for a final
# reviewed solution.
if not ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES:
    SOLUTION_STATUS = "MANUALLY_REVIEWED"
elif (len(mechanism_input) < STRICT_TARGET_POLARITIES
      or gap_final >= STRICT_MAX_AZIMUTH_GAP_DEG
      or (mechanism_input["selection_source"] == "fallback_auto_for_coverage").any()):
    SOLUTION_STATUS = "EXPLORATORY_UNDERCONSTRAINED_AUTO"
else:
    SOLUTION_STATUS = "PROVISIONAL_AUTO_UNLESS_REVIEWED"

print("Solution status:", SOLUTION_STATUS)
print("Strict polarities before fallback:", initial_n)
print(f"Initial azimuth gap: {initial_gap:.1f}°")
print("Fallback polarities added:", int((mechanism_input["selection_source"] == "fallback_auto_for_coverage").sum()))
print("Polarities entering inversion:", len(mechanism_input))
print(f"Final inversion azimuth gap: {gap_final:.1f}°")

display_cols = [
    "network", "station", "channel", "epicentral_km", "azimuth_deg",
    "takeoff_angle_deg", "polarity", "reviewed_bool", "snr", "confidence",
    "weight", "selection_source"
]
display(mechanism_input[[c for c in display_cols if c in mechanism_input.columns]])

# -----------------------------------------------------------------------------
# 4) Safety gates: warn by default; optionally restore hard-stop behavior.
# -----------------------------------------------------------------------------
if len(mechanism_input) < MIN_EXPLORATORY_POLARITIES:
    raise RuntimeError(
        f"Only {len(mechanism_input)} usable polarities remain. "
        f"At least {MIN_EXPLORATORY_POLARITIES} are required even for an "
        "exploratory double-couple grid search."
    )

underconstrained_reasons = []
if len(mechanism_input) < STRICT_TARGET_POLARITIES:
    underconstrained_reasons.append(
        f"only {len(mechanism_input)} polarities (<{STRICT_TARGET_POLARITIES})"
    )
if gap_final >= STRICT_MAX_AZIMUTH_GAP_DEG:
    underconstrained_reasons.append(
        f"azimuth gap {gap_final:.1f}° (>= {STRICT_MAX_AZIMUTH_GAP_DEG:.0f}°)"
    )
if (mechanism_input["selection_source"] == "fallback_auto_for_coverage").any():
    underconstrained_reasons.append("lower-confidence automatic fallback polarities were used")

if underconstrained_reasons:
    msg = (
        "FOCAL-MECHANISM WARNING: " + "; ".join(underconstrained_reasons) + ". "
        "The following strike/dip/rake and Coulomb maps are exploratory only. "
        "For the final scientific solution, visually review POLARITY_REVIEW.csv "
        "and re-run with ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES=False."
    )
    warnings.warn(msg)

    if STOP_IF_UNDERCONSTRAINED:
        raise RuntimeError(msg)

## 10. Optional export to the supplied SeismicDesk `polarities.csv` format

This cell does **not** overwrite anything unless you set `WRITE_SEISMICDESK_POLARITIES=True`.

For a final run, export only manually reviewed rows.

In [ ]:
WRITE_SEISMICDESK_POLARITIES = False

if WRITE_SEISMICDESK_POLARITIES:
    export = mechanism_input.copy()

    if not export["reviewed_bool"].all():
        raise RuntimeError(
            "Refusing to write SeismicDesk polarities.csv because some rows are not manually reviewed."
        )

    out_pol = pd.DataFrame({
        "network": export["network"],
        "station": export["station"],
        "polarity": export["polarity"],
        "azimuth_deg": export["azimuth_deg"],
        "takeoff_angle_deg": export["takeoff_angle_deg"],
        "reviewed": True,
    })

    target = ANALYSIS_ROOT / "inputs" / "polarities.csv"
    out_pol.to_csv(target, index=False)
    print("Wrote:", target)
else:
    print("SeismicDesk polarities.csv was not modified.")

# Part IV — focal-mechanism inversion

## 11. Double-couple radiation equations

This section intentionally follows the same coordinate convention and first-motion formulation as the `focal()` function already included in your supplied `analysis/run_analysis.py`.

For each strike/dip/rake candidate:

\[
A_P = 2(\mathbf{r}\cdot\mathbf{n})(\mathbf{r}\cdot\mathbf{s})
\]

where:
- \(\mathbf{r}\) is the ray direction from station azimuth and takeoff angle;
- \(\mathbf{n}\) is the fault normal;
- \(\mathbf{s}\) is the slip vector.

The sign of \(A_P\) is compared with observed compression/dilatation.

In [ ]:
def rays_from_obs(obs):
    az = np.deg2rad(obs["azimuth_deg"].to_numpy(float))
    take = np.deg2rad(obs["takeoff_angle_deg"].to_numpy(float))
    return np.array([
        np.sin(take)*np.cos(az),
        np.sin(take)*np.sin(az),
        np.cos(take),
    ]).T

RAYS = rays_from_obs(mechanism_input)
OBS_POL = mechanism_input["polarity"].to_numpy(int)
OBS_W = mechanism_input["weight"].to_numpy(float)

def normal_slip(strike, dip, rake):
    s = np.deg2rad(strike)
    d = np.deg2rad(dip)
    r = np.deg2rad(rake)

    normal = np.array([
        -np.sin(d)*np.sin(s),
         np.sin(d)*np.cos(s),
        -np.cos(d)
    ])

    slip = np.array([
        np.cos(r)*np.cos(s) + np.cos(d)*np.sin(r)*np.sin(s),
        np.cos(r)*np.sin(s) - np.cos(d)*np.sin(r)*np.cos(s),
        -np.sin(r)*np.sin(d)
    ])

    return normal, slip

def focal_misfit(strike, dip, rake):
    normal, slip = normal_slip(strike, dip, rake)
    radiation = 2.0*(RAYS @ normal)*(RAYS @ slip)

    pred = np.where(radiation >= 0, 1, -1)
    wrong = pred != OBS_POL

    weighted = float(np.sum(OBS_W*wrong)/np.sum(OBS_W))
    unweighted = float(np.mean(wrong))

    return weighted, unweighted, int(np.sum(wrong)), pred, radiation

## 12. Coarse 10° search

The supplied SeismicDesk focal module itself uses a 10° grid.  
We first reproduce that resolution, then refine the best region to 2°.

In [ ]:
coarse_rows = []

for strike in tqdm(range(0, 360, 10), desc="Coarse strike"):
    for dip in range(10, 91, 10):
        for rake in range(-180, 180, 10):
            wm, um, nw, _, _ = focal_misfit(strike, dip, rake)
            coarse_rows.append({
                "strike": strike,
                "dip": dip,
                "rake": rake,
                "weighted_misfit": wm,
                "misfit_fraction": um,
                "polarity_errors": nw,
            })

coarse = pd.DataFrame(coarse_rows).sort_values(
    ["weighted_misfit","polarity_errors"]
).reset_index(drop=True)

coarse.to_csv(TABLES / "focal_coarse_10deg.csv", index=False)

print("Best coarse candidates:")
display(coarse.head(30))

## 13. Refine the best coarse families to 2°

Instead of refining only one coarse solution, the notebook refines several best seeds because first-motion mechanisms can be non-unique.

In [ ]:
N_COARSE_SEEDS = 12
FINE_STEP = 2

def circular_values(center, half_width, step):
    x = np.arange(center-half_width, center+half_width+0.1*step, step)
    return np.unique(np.mod(x, 360).astype(float))

fine_tested = {}
seeds = coarse.head(N_COARSE_SEEDS)

for _, seed in tqdm(seeds.iterrows(), total=len(seeds), desc="Fine seed families"):
    strikes = circular_values(seed.strike, 12, FINE_STEP)
    dips = np.arange(max(2, seed.dip-12), min(90, seed.dip+12)+0.1, FINE_STEP)
    rakes = np.arange(max(-180, seed.rake-18), min(178, seed.rake+18)+0.1, FINE_STEP)

    for st in strikes:
        for di in dips:
            for ra in rakes:
                key = (float(st), float(di), float(ra))
                if key in fine_tested:
                    continue

                wm, um, nw, _, _ = focal_misfit(st, di, ra)
                fine_tested[key] = {
                    "strike": st,
                    "dip": di,
                    "rake": ra,
                    "weighted_misfit": wm,
                    "misfit_fraction": um,
                    "polarity_errors": nw,
                }

fine = pd.DataFrame(fine_tested.values()).sort_values(
    ["weighted_misfit","polarity_errors"]
).reset_index(drop=True)

fine.to_csv(TABLES / "focal_refined_2deg.csv", index=False)

best = fine.iloc[0]
BEST_STRIKE = float(best["strike"])
BEST_DIP = float(best["dip"])
BEST_RAKE = float(best["rake"])

print("BEST WAVEFORM-DERIVED NP1")
print(f"Strike = {BEST_STRIKE:.1f}°")
print(f"Dip    = {BEST_DIP:.1f}°")
print(f"Rake   = {BEST_RAKE:.1f}°")
print(f"Weighted misfit = {best['weighted_misfit']:.3f}")
print(f"Polarity errors = {int(best['polarity_errors'])}/{len(mechanism_input)}")

display(fine.head(30))

## 14. Auxiliary nodal plane NP2

In [ ]:
NP2_STRIKE, NP2_DIP, NP2_RAKE = aux_plane(
    BEST_STRIKE, BEST_DIP, BEST_RAKE
)

nodal_planes = pd.DataFrame([
    {
        "plane": "NP1_waveform",
        "strike": BEST_STRIKE,
        "dip": BEST_DIP,
        "rake": BEST_RAKE,
        "solution_status": SOLUTION_STATUS,
    },
    {
        "plane": "NP2_auxiliary",
        "strike": float(NP2_STRIKE),
        "dip": float(NP2_DIP),
        "rake": float(NP2_RAKE),
        "solution_status": SOLUTION_STATUS,
    },
])

nodal_planes.to_csv(TABLES / "nodal_planes.csv", index=False)
display(nodal_planes)

## 15. Best-fit observed/predicted polarity table

In [ ]:
wm, um, nw, pred, radiation = focal_misfit(
    BEST_STRIKE, BEST_DIP, BEST_RAKE
)

fit = mechanism_input.copy()
fit["predicted_polarity"] = pred
fit["radiation_amplitude"] = radiation
fit["correct"] = fit["predicted_polarity"] == fit["polarity"]

fit.to_csv(TABLES / "best_mechanism_polarity_fit.csv", index=False)

display(
    fit[
        ["network","station","azimuth_deg","takeoff_angle_deg",
         "polarity","predicted_polarity","correct","confidence"]
    ].sort_values("azimuth_deg")
)

## 16. Near-best solution family

This provides a simple non-uniqueness diagnostic.

For the final paper, consider a stronger uncertainty treatment (e.g. SKHASH and/or regional waveform moment tensor inversion).

In [ ]:
NEAR_BEST_DELTA = 0.08

near = fine[
    fine["weighted_misfit"]
    <= fine.iloc[0]["weighted_misfit"] + NEAR_BEST_DELTA
].copy()

near.to_csv(TABLES / "near_best_focal_solutions.csv", index=False)

print("Near-best solutions:", len(near))
print("Best weighted misfit:", fine.iloc[0]["weighted_misfit"])
print("Dip range:", near["dip"].min(), "to", near["dip"].max())
print("Rake range:", near["rake"].min(), "to", near["rake"].max())

display(near.head(50))

## 17. Beachball

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

bb = beach(
    [BEST_STRIKE, BEST_DIP, BEST_RAKE],
    xy=(0,0),
    width=180,
    linewidth=1.2,
    facecolor="0.35"
)

ax.add_collection(bb)
ax.set_xlim(-110,110)
ax.set_ylim(-110,110)
ax.set_aspect("equal")
ax.axis("off")

ax.set_title(
    f"Hilvan M5.3 waveform-derived focal mechanism\n"
    f"NP1 {BEST_STRIKE:.0f}/{BEST_DIP:.0f}/{BEST_RAKE:.0f} | "
    f"NP2 {NP2_STRIKE:.0f}/{NP2_DIP:.0f}/{NP2_RAKE:.0f}\n"
    f"{SOLUTION_STATUS}; polarity misfit {um:.2f}"
)

fig.savefig(FIGS / "focal_mechanism_beachball.png", dpi=250, bbox_inches="tight")
plt.show()

# Part V — Coulomb failure stress

## 18. Source-size assumptions

Your supplied `analysis/inputs/source_model.json` does not yet contain the physical source parameters required for an independent spectral stress-drop solution.

Therefore the first Coulomb model uses:

- event magnitude 5.3 as an **exploratory Mw proxy**;
- shear modulus = 32 GPa;
- Poisson ratio = 0.25;
- assumed stress drop = 3 MPa;
- rectangular aspect ratio = 2:1;
- effective friction = 0.4;
- pore pressure change = 0.

Change these values when a reviewed Mw, spectral stress drop, finite fault or geodetic source model becomes available.

In [ ]:
# ----- Coulomb assumptions -----
SOURCE_MW = EVENT_MAG
SHEAR_MODULUS_PA = 32e9
POISSON = 0.25

STRESS_DROP_MPA = 3.0
ASPECT_RATIO = 2.0

EFFECTIVE_FRICTION = 0.4
PORE_PRESSURE_CHANGE_PA = 0.0

RECEIVER_DEPTH_KM = EVENT_DEPTH_KM

CFS_HALF_WIDTH_KM = 80.0
CFS_GRID_N = 181
# -------------------------------

def mw_to_m0(mw):
    return 10.0**(1.5*mw + 9.1)

def estimate_rectangular_source(
    mw,
    stress_drop_mpa=STRESS_DROP_MPA,
    aspect_ratio=ASPECT_RATIO,
    mu_pa=SHEAR_MODULUS_PA
):
    m0 = mw_to_m0(mw)
    ds = stress_drop_mpa*1e6

    # circular crack relation
    radius = (7.0*m0/(16.0*ds))**(1.0/3.0)
    area = np.pi*radius**2

    length = np.sqrt(area*aspect_ratio)
    width = area/length
    slip = m0/(mu_pa*area)

    return {
        "M0_Nm": m0,
        "stress_drop_MPa": stress_drop_mpa,
        "equivalent_radius_km": radius/1000.0,
        "area_km2": area/1e6,
        "length_km": length/1000.0,
        "width_km": width/1000.0,
        "mean_slip_m": slip,
    }

source_scale = estimate_rectangular_source(SOURCE_MW)
display(pd.DataFrame([source_scale]))

## 19. Load Pyrocko Okada

In [ ]:
try:
    from pyrocko.modelling import OkadaSource, okada_ext
except Exception as exc:
    raise ImportError(
        "Pyrocko is required for the Coulomb section. "
        "On Windows/Conda install with: conda install -c pyrocko pyrocko"
    ) from exc

## 20. Stress tensor and ΔCFS

\[
\Delta CFS = \Delta\tau + \mu'(\Delta\sigma_n + \Delta P)
\]

The default receiver plane for each scenario is the same orientation as its source nodal plane.

This is useful for comparing NP1 versus NP2, but it is **not yet a mapped-fault receiver analysis**.

In [ ]:
def lame_lambda(mu, nu):
    return 2.0*mu*nu/(1.0-2.0*nu)

def receiver_vectors(strike, dip, rake):
    ph = np.deg2rad(strike)
    de = np.deg2rad(dip)
    ra = np.deg2rad(rake)

    ns = np.zeros(3)
    rst = np.zeros(3)
    rdi = np.zeros(3)

    ns[0] = np.sin(de)*np.cos(ph + 0.5*np.pi)
    ns[1] = np.sin(de)*np.sin(ph + 0.5*np.pi)
    ns[2] = -np.cos(de)

    rst[0] = np.cos(ph)
    rst[1] = np.sin(ph)
    rst[2] = 0.0

    rdi[0] = np.cos(de)*np.cos(ph + 0.5*np.pi)
    rdi[1] = np.cos(de)*np.sin(ph + 0.5*np.pi)
    rdi[2] = np.sin(de)

    ts = rst*np.cos(ra) - rdi*np.sin(ra)
    return ns, ts

def okada_cfs(
    source_strike,
    source_dip,
    source_rake,
    receiver_strike=None,
    receiver_dip=None,
    receiver_rake=None,
    receiver_depth_km=RECEIVER_DEPTH_KM,
    stress_drop_mpa=STRESS_DROP_MPA,
    friction=EFFECTIVE_FRICTION,
    half_width_km=CFS_HALF_WIDTH_KM,
    grid_n=CFS_GRID_N,
):
    if receiver_strike is None:
        receiver_strike = source_strike
        receiver_dip = source_dip
        receiver_rake = source_rake

    scale = estimate_rectangular_source(
        SOURCE_MW,
        stress_drop_mpa=stress_drop_mpa
    )

    L = scale["length_km"]*1000.0
    W = scale["width_km"]*1000.0
    D = scale["mean_slip_m"]

    source = OkadaSource(
        lat=EVENT_LAT,
        lon=EVENT_LON,
        north_shift=0.0,
        east_shift=0.0,
        depth=EVENT_DEPTH_KM*1000.0,
        al1=-L/2.0,
        al2=L/2.0,
        aw1=-W/2.0,
        aw2=W/2.0,
        strike=float(source_strike),
        dip=float(source_dip),
        rake=float(source_rake),
        slip=float(D),
        opening=0.0,
        poisson=POISSON,
        shearmod=SHEAR_MODULUS_PA,
    )

    n = np.linspace(-half_width_km, half_width_km, grid_n)*1000.0
    e = np.linspace(-half_width_km, half_width_km, grid_n)*1000.0
    EE, NN = np.meshgrid(e, n)

    rec = np.column_stack([
        NN.ravel(),
        EE.ravel(),
        np.full(EE.size, receiver_depth_km*1000.0)
    ])

    lam = lame_lambda(SHEAR_MODULUS_PA, POISSON)

    result = okada_ext.okada(
        source.source_patch()[None, :],
        source.source_disloc()[None, :],
        rec,
        lam,
        SHEAR_MODULUS_PA,
        nthreads=0,
        rotate_sdn=False,
        stack_sources=True
    )

    if result.ndim != 2 or result.shape[1] < 12:
        raise RuntimeError(f"Unexpected Okada output shape: {result.shape}")

    grad = result[:, 3:12]
    grad_T = result[:, (3,6,9,4,7,10,5,8,11)]
    strain = 0.5*(grad + grad_T)

    diag = [0,4,8]
    dil = strain[:,diag].sum(axis=1)[:,None]

    I = np.zeros(9)
    I[diag] = 1.0

    stress = I[None,:]*lam*dil + 2.0*SHEAR_MODULUS_PA*strain

    ns, ts = receiver_vectors(
        receiver_strike,
        receiver_dip,
        receiver_rake
    )

    sigma_n = np.sum(
        np.tile(ns,3)*stress*np.repeat(ns,3),
        axis=1
    )

    tau = np.sum(
        np.tile(ts,3)*stress*np.repeat(ns,3),
        axis=1
    )

    cfs = tau + friction*(sigma_n + PORE_PRESSURE_CHANGE_PA)

    return {
        "source": source,
        "scale": scale,
        "east_m": EE,
        "north_m": NN,
        "cfs_pa": cfs.reshape(grid_n,grid_n),
        "normal_pa": sigma_n.reshape(grid_n,grid_n),
        "shear_pa": tau.reshape(grid_n,grid_n),
        "receiver_sdr": (
            receiver_strike,
            receiver_dip,
            receiver_rake
        )
    }

## 21. Compute both nodal-plane Coulomb scenarios

In [ ]:
from pyproj import CRS, Transformer
from matplotlib.colors import TwoSlopeNorm

local_crs = CRS.from_proj4(
    f"+proj=aeqd +lat_0={EVENT_LAT} +lon_0={EVENT_LON} "
    "+datum=WGS84 +units=m +no_defs"
)
to_geo = Transformer.from_crs(local_crs, CRS.from_epsg(4326), always_xy=True)

scenarios = [
    ("NP1_waveform", BEST_STRIKE, BEST_DIP, BEST_RAKE),
    ("NP2_auxiliary", float(NP2_STRIKE), float(NP2_DIP), float(NP2_RAKE)),
]

cfs_results = {}

for name, strike, dip, rake in scenarios:
    print(f"\n{name}: {strike:.1f}/{dip:.1f}/{rake:.1f}")

    res = okada_cfs(strike, dip, rake)
    cfs_results[name] = res

    lon, lat = to_geo.transform(res["east_m"], res["north_m"])
    cfs_mpa = res["cfs_pa"]/1e6

    np.savez_compressed(
        CFS_DIR / f"{EVENT_ID}_{name}_CFS.npz",
        longitude=lon,
        latitude=lat,
        east_km=res["east_m"]/1000.0,
        north_km=res["north_m"]/1000.0,
        cfs_mpa=cfs_mpa,
        normal_mpa=res["normal_pa"]/1e6,
        shear_mpa=res["shear_pa"]/1e6,
        source_strike=strike,
        source_dip=dip,
        source_rake=rake,
        receiver_depth_km=RECEIVER_DEPTH_KM,
        effective_friction=EFFECTIVE_FRICTION,
        assumed_stress_drop_mpa=STRESS_DROP_MPA,
        source_length_km=res["scale"]["length_km"],
        source_width_km=res["scale"]["width_km"],
        source_mean_slip_m=res["scale"]["mean_slip_m"],
        focal_solution_status=SOLUTION_STATUS,
    )

    finite = np.isfinite(cfs_mpa)
    vmax = np.nanpercentile(np.abs(cfs_mpa[finite]), 98) if finite.any() else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(10,9))

    mesh = ax.pcolormesh(
        lon, lat, cfs_mpa,
        shading="auto",
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    )

    ax.scatter(EVENT_LON, EVENT_LAT, marker="*", s=180, label="Mainshock")

    ph = np.deg2rad(strike)
    halfL = res["scale"]["length_km"]*500.0
    nn = np.array([-halfL*np.cos(ph), halfL*np.cos(ph)])
    ee = np.array([-halfL*np.sin(ph), halfL*np.sin(ph)])
    flon, flat = to_geo.transform(ee, nn)
    ax.plot(flon, flat, linewidth=3, label=f"{name} strike")

    cb = fig.colorbar(mesh, ax=ax)
    cb.set_label("ΔCFS (MPa)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(
        f"Hilvan M5.3 — {name} Coulomb scenario\n"
        f"S/D/R={strike:.0f}/{dip:.0f}/{rake:.0f}; "
        f"receiver depth={RECEIVER_DEPTH_KM:.1f} km; μ'={EFFECTIVE_FRICTION:.2f}\n"
        f"{SOLUTION_STATUS}"
    )
    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(FIGS / f"{EVENT_ID}_{name}_CFS.png", dpi=250)
    plt.show()

## 22. NP1 versus NP2 numerical summary

In [ ]:
summary_rows = []

for name, res in cfs_results.items():
    c = res["cfs_pa"]/1e6

    summary_rows.append({
        "scenario": name,
        "min_CFS_MPa": float(np.nanmin(c)),
        "max_CFS_MPa": float(np.nanmax(c)),
        "median_CFS_MPa": float(np.nanmedian(c)),
        "p95_abs_CFS_MPa": float(np.nanpercentile(np.abs(c),95)),
        "positive_grid_fraction": float(np.mean(c>0)),
        "source_length_km": res["scale"]["length_km"],
        "source_width_km": res["scale"]["width_km"],
        "source_mean_slip_m": res["scale"]["mean_slip_m"],
    })

cfs_summary = pd.DataFrame(summary_rows)
cfs_summary.to_csv(CFS_DIR / "NP1_NP2_CFS_summary.csv", index=False)
display(cfs_summary)

## 23. Optional sensitivity ensemble

Recommended before publication.

Tests:
- stress drop: 1, 3, 5 MPa;
- effective friction: 0.2, 0.4, 0.6;
- receiver depth: 5, 10, 15 km.

In [ ]:
RUN_CFS_SENSITIVITY = False

if RUN_CFS_SENSITIVITY:
    rows = []

    for name, strike, dip, rake in scenarios:
        for ds in [1.0,3.0,5.0]:
            for mu in [0.2,0.4,0.6]:
                for dep in [5.0,10.0,15.0]:
                    res = okada_cfs(
                        strike, dip, rake,
                        receiver_depth_km=dep,
                        stress_drop_mpa=ds,
                        friction=mu,
                        grid_n=121
                    )
                    c = res["cfs_pa"]/1e6

                    rows.append({
                        "plane": name,
                        "stress_drop_MPa": ds,
                        "friction": mu,
                        "receiver_depth_km": dep,
                        "min_CFS_MPa": np.nanmin(c),
                        "max_CFS_MPa": np.nanmax(c),
                        "p95_abs_CFS_MPa": np.nanpercentile(np.abs(c),95),
                        "positive_fraction": np.mean(c>0),
                    })

    sensitivity = pd.DataFrame(rows)
    sensitivity.to_csv(CFS_DIR / "CFS_sensitivity.csv", index=False)
    display(sensitivity)
else:
    print("Sensitivity ensemble skipped.")

# Part VI — final report

## 24. Write a compact scientific summary

The output explicitly records whether the focal mechanism is still based on automatic/unreviewed polarities.

In [ ]:
final = {
    "event_id": EVENT_ID,
    "origin_time": str(EVENT_TIME),
    "latitude": EVENT_LAT,
    "longitude": EVENT_LON,
    "depth_km": EVENT_DEPTH_KM,
    "event_magnitude": EVENT_MAG,
    "download_radius_km": DOWNLOAD_RADIUS_KM,

    "velocity_model": VELOCITY_MODEL,
    "n_polarities": int(len(mechanism_input)),
    "azimuth_gap_deg": float(gap_final),

    "focal_solution_status": SOLUTION_STATUS,
    "weighted_polarity_misfit": float(wm),
    "polarity_errors": int(nw),

    "NP1_strike": BEST_STRIKE,
    "NP1_dip": BEST_DIP,
    "NP1_rake": BEST_RAKE,

    "NP2_strike": float(NP2_STRIKE),
    "NP2_dip": float(NP2_DIP),
    "NP2_rake": float(NP2_RAKE),

    "Coulomb_source_Mw_assumed": SOURCE_MW,
    "assumed_stress_drop_MPa": STRESS_DROP_MPA,
    "effective_friction": EFFECTIVE_FRICTION,
    "receiver_depth_km": RECEIVER_DEPTH_KM,

    "source_length_km": source_scale["length_km"],
    "source_width_km": source_scale["width_km"],
    "source_mean_slip_m": source_scale["mean_slip_m"],
}

final_df = pd.DataFrame([final])
final_df.to_csv(OUT / "FINAL_FOCAL_COULOMB_SUMMARY.csv", index=False)

(OUT / "FINAL_FOCAL_COULOMB_SUMMARY.json").write_text(
    json.dumps(final, indent=2),
    encoding="utf-8"
)

display(final_df.T)

## 25. Output inventory

In [ ]:
inventory_rows = []

for p in sorted(OUT.rglob("*")):
    if p.is_file():
        inventory_rows.append({
            "file": str(p.relative_to(OUT)),
            "size_kB": p.stat().st_size/1024.0
        })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(OUT / "OUTPUT_INVENTORY.csv", index=False)

display(inventory)

# Scientific checkpoints before using the Coulomb result

### Focal mechanism
- Visually review every accepted P first motion.
- Re-run with `ALLOW_UNREVIEWED_AUTOMATIC_POLARITIES=False`.
- Check polarity against any later AFAD, KOERI, USGS or other reviewed moment-tensor solution.
- The available physical-site geometry has useful but not ideal azimuth coverage; treat solution families and uncertainty seriously.
- A regional velocity model would improve takeoff-angle accuracy compared with the default global `iasp91`.

### Rupture plane
A focal mechanism gives **two nodal planes**. It does not by itself prove which one ruptured.

Use:
- mapped active-fault strike;
- relocated aftershocks;
- InSAR;
- GNSS;
- field rupture;
- a later finite-fault or moment-tensor solution

to select the physical rupture plane.

### Coulomb model
- The present source dimensions are based on an assumed Mw and stress drop, not a measured finite-fault model.
- Run NP1 and NP2 until the rupture plane is independently constrained.
- Test friction, stress drop and receiver depth.
- For fault-specific interpretation, use the mapped receiver fault's strike/dip/rake rather than automatically using the source-plane orientation.
- Positive ΔCFS is a modelled increase in failure tendency under the chosen assumptions; it is **not an earthquake prediction**.

### Stronger next stage
For a publication-grade result, compare this first-motion solution with a **regional full-waveform moment-tensor inversion** using the same downloaded three-component data.